In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlproyectomarlon";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

###Creation of external locations

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credencial`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

### Catalog creation

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_dev
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

###Schemas creation

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_dev.raw;
DROP SCHEMA IF EXISTS catalog_dev.bronze;
DROP SCHEMA IF EXISTS catalog_dev.silver;
DROP SCHEMA IF EXISTS catalog_dev.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_dev.raw;
CREATE SCHEMA IF NOT EXISTS catalog_dev.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_dev.silver;
CREATE SCHEMA IF NOT EXISTS catalog_dev.golden;

## Tables creation
#### Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.items (
ItemCode string,
ItemName string,
CategoryCode integer,
CategoryName STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/items"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.everyday_Sales (
  Date DATE,
  Time TIMESTAMP,
  ItemCode string,
  QuantitySold_kilo double,
  UnitSellingPrice integer,
  SaleOrReturn string,
  Discount BOOLEAN
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/everyday_Sales"

### Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.silver.sales_cleaned_grouped
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/sales_cleaned_grouped"
AS
SELECT
    s.Date,
    s.Time,
    s.ItemCode,
    i.ItemName,
    i.CategoryCode,
    i.CategoryName,
    s.QuantitySold_kilo,
    s.UnitSellingPrice,
    s.SaleOrReturn,
    s.Discount
FROM catalog_dev.bronze.everyday_Sales s
INNER JOIN catalog_dev.bronze.items i
ON s.ItemCode = i.ItemCode;

###Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.sales_by_category
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/sales_by_category"
AS
SELECT
    CategoryCode,
    CategoryName,
    SUM(QuantitySold_kilo) AS TotalQuantitySold,
    SUM(QuantitySold_kilo * UnitSellingPrice) AS TotalRevenue
FROM catalog_dev.silver.sales_cleaned_grouped
WHERE SaleOrReturn = 'Sale'
GROUP BY CategoryCode, CategoryName;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.top_products
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/top_products"
AS
SELECT
    ItemCode,
    ItemName,
    CategoryName,
    SUM(QuantitySold_kilo) AS TotalQuantitySold,
    SUM(QuantitySold_kilo * UnitSellingPrice) AS TotalRevenue
FROM catalog_dev.silver.sales_cleaned_grouped
WHERE SaleOrReturn = 'Sale'
GROUP BY
    ItemCode,
    ItemName,
    CategoryName
ORDER BY TotalRevenue DESC;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.daily_sales
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/daily_sales"
AS
SELECT
    Date,
    COUNT(*) AS NumberOfTransactions,
    SUM(QuantitySold_kilo) AS TotalQuantitySold,
    SUM(QuantitySold_kilo * UnitSellingPrice) AS TotalRevenue
FROM catalog_dev.silver.sales_cleaned_grouped
WHERE SaleOrReturn = 'Sale'
GROUP BY Date
ORDER BY Date;